# AgenticLens Beginner Demo: Multi-Agent Workflow

This notebook is for a first-time user who wants to understand AgenticLens with multi-agent systems.

Agents in this workflow:

1. Planner Agent
2. Retriever Agent
3. Tool Agent
4. Writer Agent
5. Reviewer Agent
6. Final Response Agent

Default mode is mock mode, so it runs without an API key.

In [ ]:
# Optional. Run only if imports fail.
# !pip install agenticlens openai pandas matplotlib

## 1. Imports

In [ ]:
import os
import time
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd

from agenticlens import profile, step
from agenticlens.exporters import JSONExporter

## 2. Choose Mock Mode or Real OpenAI Mode

In [ ]:
USE_REAL_OPENAI = False
MODEL_NAME = "gpt-4o-mini"

if USE_REAL_OPENAI:
    from openai import OpenAI

    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
    client = OpenAI()
else:
    client = None

## 3. User Question

In [ ]:
user_question = "My order A123 was delivered. Can I get a refund and how long will it take?"
print(user_question)

## 4. Knowledge Base

In [ ]:
DOCS = [
    "Refund policy: Customers can request a refund within 30 days of delivery.",
    "Refund policy: Items must be unused and in original packaging.",
    "Refunds are processed to the original payment method.",
    "Refunds may take 5 to 10 business days.",
    "Order tracking: Customers can track orders using their order ID.",
]

## 5. Tools and Retriever

In [ ]:
def clean_words(text: str):
    return set(text.lower().replace("?", "").replace(".", "").replace(",", "").split())

def retrieve_context(query: str, top_k: int = 5):
    query_words = clean_words(query)
    scored_docs = []
    for doc in DOCS:
        score = len(query_words.intersection(clean_words(doc)))
        scored_docs.append((score, doc))
    scored_docs.sort(reverse=True, key=lambda item: item[0])
    return [doc for score, doc in scored_docs[:top_k] if score > 0]

def lookup_order(order_id: str):
    time.sleep(0.2)
    return {
        "order_id": order_id,
        "status": "Delivered",
        "delivered_days_ago": 12,
        "eligible_for_refund": True,
    }

## 6. Mock and Real LLM Call

In [ ]:
class MockUsage:
    def __init__(self, prompt_tokens: int, completion_tokens: int):
        self.prompt_tokens = prompt_tokens
        self.completion_tokens = completion_tokens

class MockMessage:
    def __init__(self, content: str):
        self.content = content

class MockChoice:
    def __init__(self, content: str):
        self.message = MockMessage(content)

class MockResponse:
    def __init__(self, content: str, prompt_tokens: int, completion_tokens: int):
        self.usage = MockUsage(prompt_tokens, completion_tokens)
        self.choices = [MockChoice(content)]


def call_llm(agent_name: str, prompt: str):
    if USE_REAL_OPENAI:
        messages = [
            {
                "role": "system",
                "content": (
                    f"You are the {agent_name} in a customer support "
                    "multi-agent workflow."
                ),
            },
            {"role": "user", "content": prompt},
        ]
        return client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0,
        )

    answers = {
        "Planner Agent": (
            "Plan: retrieve refund policy, look up order A123, write an answer, "
            "review it, then send final response."
        ),
        "Writer Agent": (
            "Draft: The order was delivered and appears eligible. "
            "Refunds may take 5 to 10 business days."
        ),
        "Reviewer Agent": (
            "Review: Add that refunds go to the original payment method "
            "and mention the 30-day policy."
        ),
        "Final Response Agent": (
            "Yes, your order appears eligible for a refund if it is within 30 days "
            "and meets the return conditions. Refunds are sent to the original "
            "payment method and may take 5 to 10 business days."
        ),
    }
    tokens = {
        "Planner Agent": (160, 45),
        "Writer Agent": (360, 90),
        "Reviewer Agent": (310, 65),
        "Final Response Agent": (260, 75),
    }
    prompt_tokens, completion_tokens = tokens.get(agent_name, (100, 30))
    return MockResponse(
        answers.get(agent_name, "Mock answer"),
        prompt_tokens,
        completion_tokens,
    )

## 7. Start AgenticLens Profiling

In [ ]:
profile_context = profile("Beginner Multi-Agent Workflow")
workflow = profile_context.__enter__()
print("Started:", workflow.name)

## 8. Planner Agent

In [ ]:
with step(
    "Planner Agent",
    type="planner",
    provider="openai",
    model=MODEL_NAME,
    prompt=user_question,
) as s:
    start = time.time()
    planner_response = call_llm(
        "Planner Agent",
        "Create a short support workflow plan for: " + user_question,
    )
    s.record(planner_response)
    s.step.metrics.latency = time.time() - start
    plan = planner_response.choices[0].message.content
print(plan)

## 9. Retriever Agent

In [ ]:
with step("Retriever Agent", type="retriever", chunk_count=5, avg_tokens_per_chunk=35) as s:
    start = time.time()
    chunks = retrieve_context(user_question, top_k=5)
    s.step.metrics.latency = time.time() - start
    s.step.metadata["retrieved_chunks"] = chunks
    s.step.metadata["chunk_count"] = len(chunks)
for i, chunk in enumerate(chunks, start=1):
    print(f"{i}. {chunk}")

## 10. Tool Agent

In [ ]:
with step(
    "Tool Agent - Lookup Order",
    type="tool_call",
    provider="openai",
    model=MODEL_NAME,
    tool_name="lookup_order",
    tool_args={"order_id": "A123"},
) as s:
    start = time.time()
    order_info = lookup_order("A123")
    s.step.metrics.latency = time.time() - start
    s.step.metadata["tool_result"] = order_info
order_info

## 11. Writer Agent

In [ ]:
writer_prompt = (
    "Question: "
    + user_question
    + "\nPlan: "
    + plan
    + "\nContext: "
    + str(chunks)
    + "\nOrder info: "
    + str(order_info)
)
with step("Writer Agent", type="llm_call", provider="openai", model=MODEL_NAME) as s:
    start = time.time()
    writer_response = call_llm("Writer Agent", writer_prompt)
    s.record(writer_response)
    s.step.metrics.latency = time.time() - start
    draft_answer = writer_response.choices[0].message.content
print(draft_answer)

## 12. Reviewer Agent

In [ ]:
review_prompt = (
    "Review this draft: "
    + draft_answer
    + "\nContext: "
    + str(chunks)
    + "\nOrder info: "
    + str(order_info)
)
with step("Reviewer Agent", type="llm_call", provider="openai", model=MODEL_NAME) as s:
    start = time.time()
    reviewer_response = call_llm("Reviewer Agent", review_prompt)
    s.record(reviewer_response)
    s.step.metrics.latency = time.time() - start
    review = reviewer_response.choices[0].message.content
print(review)

## 13. Final Response Agent

In [ ]:
final_prompt = "Draft: " + draft_answer + "\nReviewer feedback: " + review
with step("Final Response Agent", type="final_response", provider="openai", model=MODEL_NAME) as s:
    start = time.time()
    final_response = call_llm("Final Response Agent", final_prompt)
    s.record(final_response)
    s.step.metrics.latency = time.time() - start
    final_answer = final_response.choices[0].message.content
print(final_answer)

## 14. Close Workflow

In [ ]:
profile_context.__exit__(None, None, None)
print("Total tokens:", workflow.total_tokens)
print("Total cost:", workflow.total_cost)

## 15. View Steps as a Table

In [ ]:
def workflow_to_dataframe(workflow):
    rows = []
    for st in workflow.steps:
        rows.append(
            {
                "step": st.name,
                "type": st.type.value if hasattr(st.type, "value") else str(st.type),
                "prompt_tokens": st.metrics.prompt_tokens or 0,
                "completion_tokens": st.metrics.completion_tokens or 0,
                "total_tokens": st.metrics.total_tokens or 0,
                "cost_usd": st.metrics.cost or 0,
                "latency_sec": st.metrics.latency or 0,
            }
        )
    return pd.DataFrame(rows)


agent_df = workflow_to_dataframe(workflow)
agent_df

## 16. Visualize Tokens by Agent

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(agent_df["step"], agent_df["total_tokens"])
plt.title("Multi-Agent Workflow: Tokens by Agent")
plt.ylabel("Total Tokens")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 17. Visualize Latency by Agent

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(agent_df["step"], agent_df["latency_sec"])
plt.title("Multi-Agent Workflow: Latency by Agent")
plt.ylabel("Latency Seconds")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 18. Save, Report, and Analyze

In [ ]:
JSONExporter().export(workflow, "beginner_multiagent_report.json")
print("Saved beginner_multiagent_report.json")

In [ ]:
!agenticlens report beginner_multiagent_report.json

In [ ]:
!agenticlens analyze beginner_multiagent_report.json

## What This Notebook Proves

AgenticLens can show which agent consumed tokens, which step took time, and where optimization may be possible.